<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 4 — Descubrimiento de temas con K-Means</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Clustering sobre los vectores TF-IDF: el mismo algoritmo de la Unidad 1, sobre texto</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Sin consulta ni etiquetas, dejar que el corpus revele su estructura temática. Construyen la matriz documento × término con TF-IDF, la normalizan, **reutilizan su K-Means de la Unidad 1** y interpretan los grupos por los términos de mayor peso de cada centroide.


## 0 · Corpus procesado del Lab 1

In [1]:
import json, math, re, unicodedata
from collections import Counter

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])
import numpy as np

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


## 1 · Matriz documento × término (TF-IDF) + normalización L2

**1.a** Construyan el vocabulario, la matriz densa de pesos TF-IDF (filas = documentos, columnas = términos) y normalicen **cada fila a norma 1**. Reutilicen su `tf`/`idf`/`tfidf` del Lab 2.

In [2]:
# Reutilizar tf, idf, tfidf del Lab 2
def tf(doc):
    if not doc:
        return {}
    counts = {}
    for t in doc:
        counts[t] = counts.get(t, 0) + 1
    n = len(doc)
    return {t: c / n for t, c in counts.items()}

def idf(corpus):
    N = len(corpus)
    df_counts = {}
    for doc in corpus:
        for t in set(doc):
            df_counts[t] = df_counts.get(t, 0) + 1
    return {t: math.log(N / df) for t, df in df_counts.items()}

def tfidf(doc, idf_):
    doc_tf = tf(doc)
    return {t: tf_val * idf_.get(t, 0.0) for t, tf_val in doc_tf.items()}

# Construir vocabulario (lista ordenada)
IDF = idf(documentos)
vocab = sorted(IDF.keys())
col = {t: i for i, t in enumerate(vocab)}
V = len(vocab)
N = len(documentos)

# Matriz densa M[doc, termino] con pesos TF-IDF
M = np.zeros((N, V))
for i, doc in enumerate(documentos):
    weights = tfidf(doc, IDF)
    for t, w in weights.items():
        if t in col:
            M[i, col[t]] = w

# Normalizar cada fila a norma L2 -> Mn
norms = np.linalg.norm(M, axis=1, keepdims=True)
norms[norms == 0] = 1.0  # evitar division por cero
Mn = M / norms

print(f'Vocabulario: {V} terminos')
print(f'Matriz TF-IDF normalizada (L2): {Mn.shape}')
print(f'Norma de la primera fila (debe ser ~1.0): {np.linalg.norm(Mn[0]):.6f}')


Vocabulario: 204 terminos
Matriz TF-IDF normalizada (L2): (14, 204)
Norma de la primera fila (debe ser ~1.0): 1.000000


> **Pregunta (defensa):** ¿por qué normalizar a L2 antes de agrupar con K-Means? Liguen su respuesta con la similitud coseno de la Sesión 1.


_Su respuesta:_
Normalizar cada fila del vector a norma L2 antes de aplicar K-Means garantiza que la distancia euclidiana entre dos documentos sea **matemáticamente equivalente a (2 − 2·cos(θ))**, donde θ es el ángulo entre sus vectores. Esto significa que cuando K-Means minimiza la distancia euclidiana al centroide, en realidad está maximizando la similitud coseno, que es la medida más relevante en recuperación de información (no queremos que documentos largos dominen el espacio simplemente por tener más términos). Sin la normalización, un documento muy largo con muchas repeticiones estaría siempre más lejos del centroide que documentos cortos, distorsionando los grupos.


## 2 · Elegir k

**2.a** Usen el **método del codo** (inercia) y el **coeficiente de silueta** para 2 ≤ k ≤ 8. Reutilicen su K-Means de la Unidad 1 para la inercia; la silueta pueden tomarla de scikit-learn (es una métrica, no el algoritmo).

In [3]:
import random
from sklearn.metrics import silhouette_score

# K-Means desde cero (reutilizamos el mismo de Unidad 1)
def kmeans(X, k, max_iter=100, seed=42):
    random.seed(seed)
    idx = random.sample(range(len(X)), k)
    centroids = X[idx].copy()
    labels = np.zeros(len(X), dtype=int)
    
    for _ in range(max_iter):
        # Asignar cada punto al centroide más cercano
        dists = np.linalg.norm(X[:, np.newaxis] - centroids[np.newaxis, :], axis=2)
        new_labels = np.argmin(dists, axis=1)
        
        if np.all(new_labels == labels):
            break
        labels = new_labels
        
        # Recalcular centroides
        for j in range(k):
            members = X[labels == j]
            if len(members) > 0:
                centroids[j] = members.mean(axis=0)
    
    # Inercia: suma de distancias cuadradas al centroide
    inertia = sum(
        np.sum((X[labels == j] - centroids[j]) ** 2)
        for j in range(k)
        if np.any(labels == j)
    )
    return labels, centroids, inertia

inertias = []
silhouettes = []
ks = range(2, 9)

for k in ks:
    labels, centroids, inertia = kmeans(Mn, k)
    inertias.append(inertia)
    sil = silhouette_score(Mn, labels)
    silhouettes.append(sil)
    print(f'k={k}  Inercia={inertia:.4f}  Silueta={sil:.4f}')

# Elegir k basado en la silueta
best_sil_idx = silhouettes.index(max(silhouettes))
K = list(ks)[best_sil_idx]
print(f'\nk elegido: K={K} (mayor coeficiente de silueta: {max(silhouettes):.4f})')


k=2  Inercia=11.8489  Silueta=0.0014
k=3  Inercia=10.7797  Silueta=0.0040
k=4  Inercia=9.7300  Silueta=0.0076
k=5  Inercia=8.8122  Silueta=-0.0018
k=6  Inercia=7.8057  Silueta=-0.0003
k=7  Inercia=6.7547  Silueta=0.0041
k=8  Inercia=5.7889  Silueta=0.0027

k elegido: K=4 (mayor coeficiente de silueta: 0.0076)


_¿Qué k eligieron y por qué? (lo defenderán):_
Elegimos el valor de **k con mayor coeficiente de silueta** obtenido en el barrido de 2 a 8. El coeficiente de silueta mide qué tan bien separados están los grupos: un valor cercano a 1 indica que los puntos están bien asignados a su cluster y lejos de los vecinos. Aunque el método del codo (inercia) es útil para detectar la inflexión donde agregar más clusters deja de aportar ganancia significativa, en corpus pequeños la curva de inercia no siempre muestra un codo claro. Por eso priorizamos la silueta como criterio objetivo.


## 3 · Agrupar e interpretar

**3.a** Corran K-Means con su k. Para **cada grupo**, listen los términos de mayor peso en el centroide y pónganle un nombre temático.

In [4]:
labels_final, centroids_final, _ = kmeans(Mn, K)

# Nombres temáticos por grupo (se asignarán tras ver los términos top)
nombres_grupos = {}

for cluster_id in range(K):
    centroide = centroids_final[cluster_id]
    
    # Top 8 términos del centroide por peso
    top_indices = np.argsort(centroide)[::-1][:8]
    top_terms = [vocab[i] for i in top_indices if centroide[i] > 0]
    
    # Documentos miembros
    members = [(corpus[i]['id'], corpus[i]['titulo']) for i in range(N) if labels_final[i] == cluster_id]
    
    # Asignar nombre temático basado en los términos dominantes
    if top_terms:
        nombres_grupos[cluster_id] = ', '.join(top_terms[:3])
    
    print(f'\n=== Cluster {cluster_id} — Tema: [{nombres_grupos.get(cluster_id, "")}] ===')
    print(f'  Términos top del centroide: {top_terms}')
    print(f'  Documentos ({len(members)}):')
    for doc_id, titulo in members:
        print(f'    - {doc_id}: {titulo}')



=== Cluster 0 — Tema: [artesanal, mercado, produccion] ===
  Términos top del centroide: ['artesanal', 'mercado', 'produccion', 'premium', 'origen', 'cooperativa', 'baja', 'chocolate']
  Documentos (3):
    - d08: Repunta la produccion de cacao
    - d09: San Cristobal, destino cultural
    - d11: Alertan por casos de dengue

=== Cluster 1 — Tema: [sequia, afectar, apoyo] ===
  Términos top del centroide: ['sequia', 'afectar', 'apoyo', 'perdida', 'estatal', 'cultivo', 'gravemente', 'frijol']
  Documentos (2):
    - d02: Crisis hidrica golpea la region
    - d04: Sequia afecta cultivos de maiz

=== Cluster 2 — Tema: [agua, pidio, colonia] ===
  Términos top del centroide: ['agua', 'pidio', 'colonia', 'tuxtla', 'seguridad', 'reducir', 'obra', 'rehabilitacion']
  Documentos (3):
    - d01: Lluvias provocan inundaciones en Tuxtla
    - d10: Avanza obra de infraestructura carretera
    - d13: Restablecen servicio de agua potable

=== Cluster 3 — Tema: [chiapa, nacional, productor] ===
  Té

## 4 · Evaluar el descubrimiento

**4.a** Localicen un documento **mal agrupado** y expliquen por qué. Pista dirigida: revisen dónde cayó **d02 (“crisis hídrica”)** y **d13 (“agua potable”)** — ¿quedaron juntos?

In [5]:
# Imprimir el cluster de d02, d13 y d04
for target_id in ['d02', 'd13', 'd04']:
    idx_doc = next((i for i, d in enumerate(corpus) if d['id'] == target_id), None)
    if idx_doc is not None:
        cluster_asignado = labels_final[idx_doc]
        print(f'{target_id} ("{corpus[idx_doc]["titulo"]}") -> Cluster {cluster_asignado}')

cluster_d02 = labels_final[next(i for i, d in enumerate(corpus) if d['id'] == 'd02')]
cluster_d13 = labels_final[next(i for i, d in enumerate(corpus) if d['id'] == 'd13')]

if cluster_d02 == cluster_d13:
    print('\n>>> d02 y d13 quedaron en el MISMO cluster (correcto semánticamente: ambos tratan agua/hídrico).')
else:
    print('\n>>> d02 y d13 quedaron en clusters DISTINTOS — falla de sinonimia en el espacio TF-IDF.')


d02 ("Crisis hidrica golpea la region") -> Cluster 1
d13 ("Restablecen servicio de agua potable") -> Cluster 2
d04 ("Sequia afecta cultivos de maiz") -> Cluster 1

>>> d02 y d13 quedaron en clusters DISTINTOS — falla de sinonimia en el espacio TF-IDF.


_¿Qué documento quedó mal agrupado y por qué? Conecten con la falla de significado de la Sesión 2:_
**d02 ('Crisis hídrica golpea la región')** es temáticamente sobre el problema del agua, igual que **d13 ('Restablecen servicio de agua potable')**. Sin embargo, el modelo TF-IDF los representa como vectores muy distintos porque utilizan vocabularios casi sin intersección:
- d02 usa términos como *'hidrico'*, *'desabasto'*, *'liquido'*, *'sequia'*, *'pozos'*.
- d13 usa términos como *'agua'*, *'potable'*, *'red'*, *'escalonada'*.
Como TF-IDF no captura el significado semántico —solo la co-ocurrencia superficial de palabras—, K-Means puede separarlos en clusters distintos porque sus vectores TF-IDF están lejos en el espacio euclidiano. Esta es exactamente la **'falla de significado'** de la Sesión 2: dos textos sobre el mismo tema quedan alejados en el espacio de representación porque no comparten palabras exactas. La solución vendría de usar **word embeddings** (por ejemplo, Word2Vec o BERT), donde 'agua', 'hídrico', 'líquido vital' y 'desabasto' convergen a zonas similares del espacio vectorial.


## Entregables — Lab 4
- [ ] Matriz TF-IDF documento × término normalizada a L2.
- [ ] Curvas de codo y silueta + **k elegido y justificado**.
- [ ] K-Means corrido + cada grupo nombrado por los términos de su centroide.
- [ ] Análisis del documento mal agrupado (caso d02/d13) conectado con embeddings.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
